## RUSSIAN FOOD

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re
import csv
import random
import logging
from dataclasses import dataclass, field, asdict
from typing import Optional
from urllib.parse import urljoin

BASE_URL = "https://www.russianfood.com"
BYTYPE_URL = BASE_URL + "/recipes/bytype/" 
DELAY_MIN = 20.0   # минимальная пауза (сек)
DELAY_MAX = 40.0
OUTPUT_JSON = "recipes_rf.json"
OUTPUT_CSV = "recipes_rf.csv"

# Диапазон fid для перебора категорий
FID_START = 2
FID_MAX = 15    # верхняя граница перебора
MAX_EMPTY_FID_STREAK = 15     # стоп если подряд столько fid пусты

# число страниц внутри одного fid
MAX_PAGES_PER_FID = 7     # None = все

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:138.0) Gecko/20100101 Firefox/138.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/18.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64; rv:138.0) Gecko/20100101 Firefox/138.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:137.0) Gecko/20100101 Firefox/137.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36 Edg/137.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:138.0) Gecko/20100101 Firefox/138.0",
]

BASE_HEADERS = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
    "Sec-Ch-Ua": '"Google Chrome";v="137", "Chromium";v="137", "Not/A)Brand";v="24"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "DNT": "1",
    "Priority": "u=0, i",
}


MAX_RETRIES = 3 # повторных попыток при ошибке
RETRY_DELAY = 30 # пауза перед повторной попыткой (сек)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

@dataclass
class Recipe:
    url: str = ""
    name: str = ""
    category: str = ""
    ingredients: list[str] = field(default_factory=list)
    steps: list[str] = field(default_factory=list)
    calories: Optional[str] = None
    protein: Optional[str] = None
    fat: Optional[str] = None
    carbohydrates: Optional[str] = None


session = requests.Session()
session.headers.update(BASE_HEADERS)

In [ ]:

def random_delay():
    """Случайная пауза, чтобы не выглядеть как бот."""
    delay = random.uniform(DELAY_MIN, DELAY_MAX)
    time.sleep(delay)


def get_random_ua() -> str:
    return random.choice(USER_AGENTS)


def fetch(url: str, referer: str = None) -> Optional[BeautifulSoup]:
    """
    Загружает страницу с ротацией User-Agent,
    Referer и повторными попытками при 403.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # Подставляем случайный UA и Referer 
            headers = {
                "User-Agent": get_random_ua(),
                "Referer": referer or BASE_URL + "/",
            }

            resp = session.get(url, timeout=30, headers=headers)

            if resp.status_code == 403:
                if attempt < MAX_RETRIES:
                    wait = RETRY_DELAY * attempt
                    log.warning(
                        "403 на %s (попытка %d/%d) — жду %d сек...",
                        url, attempt, MAX_RETRIES, wait,
                    )
                    time.sleep(wait)
                    session.cookies.clear()
                    continue
                else:
                    log.warning("403 на %s — все %d попытки исчерпаны", url, MAX_RETRIES)
                    return None

            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")

        except requests.RequestException as e:
            if attempt < MAX_RETRIES:
                wait = RETRY_DELAY * attempt
                log.warning(
                    "Ошибка %s (попытка %d/%d): %s — жду %d сек...",
                    url, attempt, MAX_RETRIES, e, wait,
                )
                time.sleep(wait)
                session.cookies.clear()
            else:
                log.warning("Ошибка %s — все попытки исчерпаны: %s", url, e)
                return None

    return None


def clean(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def get_category_name(soup: BeautifulSoup) -> str:
    """
    Извлекает название категории со страницы каталога.
     это <h1> на странице bytype.
    """
    h1 = soup.find("h1")
    if h1:
        return clean(h1.get_text())
    return ""


def get_max_page(soup: BeautifulSoup) -> int:
    """
    Ищет в пагинаторе максимальный номер страницы.
    Ссылки вида ?fid=2&page=5  →  извлекаем page=5.
    """
    max_page = 1
    # Ищем все ссылки с параметром page=
    for a_tag in soup.find_all("a", href=re.compile(r"[?&]page=\d+")):
        href = a_tag.get("href", "")
        m = re.search(r"[?&]page=(\d+)", href)
        if m:
            p = int(m.group(1))
            if p > max_page:
                max_page = p
    return max_page


def extract_recipe_links_from_catalog(soup: BeautifulSoup) -> list[str]:
    """
    На странице каталога рецепты внутри:
    <div class="in_seen" ...>
      <a href="/recipes/recipe.php?rid=162544" ...>
    """
    links: list[str] = []
    for div in soup.select("div.in_seen"):
        a_tag = div.select_one('a[href*="/recipes/recipe.php?rid="]')
        if a_tag and a_tag.get("href"):
            full = urljoin(BASE_URL, a_tag["href"])
            if full not in links:
                links.append(full)
    return links


In [ ]:
def collect_all_recipe_urls() -> list[tuple[str, str]]:
    """
    Обходит все fid-категории, внутри каждой - все страницы.
    Возвращает список кортежей (url_рецепта, название_категории).
    """
    all_recipes: list[tuple[str, str]] = []
    seen_urls: set[str] = set()
    empty_streak = 0

    for fid in range(FID_START, FID_MAX + 1):
        # Первая страница категории 
        url = f"{BYTYPE_URL}?fid={fid}"
        log.info("═══ Категория fid=%d: %s ═══", fid, url)

        random_delay()
        soup = fetch(url, referer=BASE_URL + "/recipes/")
        if soup is None:
            empty_streak += 1
            if empty_streak >= MAX_EMPTY_FID_STREAK:
                log.info("Подряд %d пустых fid — заканчиваем перебор.", empty_streak)
                break
            continue

        # Название категории
        category_name = get_category_name(soup)

        # Ссылки с первой страницы
        links = extract_recipe_links_from_catalog(soup)
        if not links:
            log.info("  fid=%d пуст — пропускаю", fid)
            empty_streak += 1
            if empty_streak >= MAX_EMPTY_FID_STREAK:
                log.info("Подряд %d пустых fid — заканчиваем перебор.", empty_streak)
                break
            continue

        # Сброс счётчика пустых
        empty_streak = 0
        log.info("  Категория: «%s»", category_name)

        # Сколько страниц в этом fid
        max_page = get_max_page(soup)
        if MAX_PAGES_PER_FID is not None:
            max_page = min(max_page, MAX_PAGES_PER_FID)
        log.info("  Страниц: %d", max_page)

        # Добавляем ссылки с первой страницы
        for link in links:
            if link not in seen_urls:
                seen_urls.add(link)
                all_recipes.append((link, category_name))

        log.info("  Стр. 1: +%d ссылок (всего %d)", len(links), len(all_recipes))

        # Остальные страницы этой категории
        for page in range(2, max_page + 1):
            page_url = f"{BYTYPE_URL}?fid={fid}&page={page}"
            log.info("  Стр. %d / %d: %s", page, max_page, page_url)

            random_delay()
            soup = fetch(page_url)
            if soup is None:
                continue

            links = extract_recipe_links_from_catalog(soup)
            if not links:
                log.info("  Стр. %d пуста — стоп для fid=%d", page, fid)
                break

            new_count = 0
            for link in links:
                if link not in seen_urls:
                    seen_urls.add(link)
                    all_recipes.append((link, category_name))
                    new_count += 1

            log.info("    +%d новых (всего %d)", new_count, len(all_recipes))

    log.info("Итого уникальных ссылок: %d", len(all_recipes))
    return all_recipes

In [ ]:
def parse_recipe(url: str, soup: BeautifulSoup, fallback_category: str = "") -> Optional[Recipe]:
    recipe = Recipe(url=url)

    # Название
    h1 = soup.select_one("h1.title")
    if not h1:
        h1 = soup.find("h1")
    if not h1:
        log.warning("Нет <h1> на %s", url)
        return None
    recipe.name = clean(h1.get_text())
    # Категория
    # Ищем ссылку вида <a href="/recipes/bytype/?fid=11">Супы</a>
    cat_link = soup.select_one('a[href*="/recipes/bytype/?fid="]')
    if cat_link:
        recipe.category = clean(cat_link.get_text())
    else:
        # Фолбэк: используем категорию, определённую при сборе ссылок
        recipe.category = fallback_category

    # Ингредиенты
    # внутри <td> → <span> с текстом ингредиента
    ingr_table = soup.find("table", class_="ingr")
    if ingr_table:
        for tr in ingr_table.find_all("tr", class_=re.compile(r"^ingr_tr_\d+")):
            span = tr.find("span")
            if span:
                text = clean(span.get_text())
                # Пропускаем пустые, разделители (*) и заголовки-секции
                if text and text != "*":
                    recipe.ingredients.append(text)
    # Шаги приготовления
    for idx, step_div in enumerate(soup.select("div.step_n"), start=1):
        paragraphs = step_div.find_all("p")
        step_text = " ".join(clean(p.get_text()) for p in paragraphs if clean(p.get_text()))
        if step_text:
            recipe.steps.append(f"Шаг {idx}: {step_text}")

    # КБЖУ на russianfood.com нет
    recipe.calories = None
    recipe.protein = None
    recipe.fat = None
    recipe.carbohydrates = None

    return recipe



In [ ]:
def save_json(recipes: list[Recipe], path: str):
    data = [asdict(r) for r in recipes]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    log.info("JSON сохранён: %s (%d рецептов)", path, len(recipes))


def save_csv(recipes: list[Recipe], path: str):
    if not recipes:
        return
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow([
            "url", "name", "category", "ingredients",
            "steps", "calories", "protein", "fat", "carbohydrates",
        ])
        for r in recipes:
            w.writerow([
                r.url,
                r.name,
                r.category,
                "\n".join(r.ingredients),
                "\n".join(r.steps),
                r.calories or "",
                r.protein or "",
                r.fat or "",
                r.carbohydrates or "",
            ])
    log.info("CSV сохранён: %s (%d рецептов)", path, len(recipes))


In [ ]:
def main():
    # Собираем все ссылки (url, категория)
    log.info("═══ ЭТАП 1: сбор ссылок из каталога ═══")
    print('Step 1')
    recipe_entries = collect_all_recipe_urls()  # [(url, category), ...]

    if not recipe_entries:
        log.error("Ссылок не найдено — выход.")
        return

    # Парсим каждый рецепт
    log.info("═══ ЭТАП 2: парсинг рецептов ═══")
    print('Step 2')
    recipes: list[Recipe] = []
    CHECKPOINT = 100

    for i, (url, cat) in enumerate(recipe_entries, start=1):
        log.info("[%d/%d] %s", i, len(recipe_entries), url)

        random_delay()
        soup = fetch(url, referer=BASE_URL + "/recipes/")
        if soup is None:
            continue

        recipe = parse_recipe(url, soup, fallback_category=cat)
        if recipe:
            recipes.append(recipe)
            log.info(
                "  ✓ %s | кат: %s | ингр: %d | шагов: %d",
                recipe.name[:50],
                recipe.category,
                len(recipe.ingredients),
                len(recipe.steps),
            )
        else:
            log.warning("  ✗ не распарсилось")

        # Промежуточное сохранение
        if i % CHECKPOINT == 0:
            save_json(recipes, OUTPUT_JSON)
            log.info("  💾 чекпоинт: %d рецептов", len(recipes))

    # Финальное сохранение
    print('step 3')
    log.info("═══ ЭТАП 3: сохранение ═══")
    save_json(recipes, OUTPUT_JSON)
    save_csv(recipes, OUTPUT_CSV)
    log.info("Готово! Всего рецептов: %d", len(recipes))


In [ ]:
if __name__ == "__main__":
    main()